step 1: importing necessary libraries

In [37]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

step 2- load dataset

In [4]:
from google.colab import files

uploaded = files.upload()

# Load the uploaded Excel file into a pandas DataFrame
file_name = list(uploaded.keys())[0]
df = pd.read_excel(file_name)

Saving VideoGames_Sales.xlsx to VideoGames_Sales.xlsx


In [6]:
df.head()

,title,console,genre,publisher,developer,critic_score,total_sales(mil),na_sales(mil),jp_sales(mil),pal_sales(mil),other_sales(mil),release_date
0,Grand Theft Auto V,PS3,Action,Rockstar Games,Rockstar North,9.4,20.32,6.37,0.99,9.85,3.12,2013-09-17
1,Grand Theft Auto V,PS4,Action,Rockstar Games,Rockstar North,9.7,19.39,6.06,0.60,9.71,3.02,2014-11-18
2,Grand Theft Auto: Vice City,PS2,Action,Rockstar Games,Rockstar North,9.6,16.15,8.41,0.47,5.49,1.78,2002-10-28
3,Grand Theft Auto V,X360,Action,Rockstar Games,Rockstar North,NaN,15.86,9.06,0.06,5.33,1.42,2013-09-17
4,Call of Duty: Black Ops 3,PS4,Shooter,Activision,Treyarch,8.1,15.09,6.18,0.41,6.05,2.44,2015-11-06


step 3 - select features and target

In [14]:
TARGET_COLUMN = 'genre'
FEATURE_COLUMNS = ['console', 'critic_score']
CATEGORICAL_FEATURES = ['console']
NUMERICAL_FEATURES = ['critic_score']


In [25]:
df.dropna(subset=[TARGET_COLUMN], inplace=True)

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

# Ensure 'console' column is of string type for OneHotEncoder
X['console'] = X['console'].astype(str)

display(X.head())
display(y.head())

,console,critic_score
0,PS3,9.4
1,PS4,9.7
2,PS2,9.6
3,X360,NaN
4,PS4,8.1


,genre
0,Action
1,Action
2,Action
3,Action
4,Shooter


step 3- Target encoding

In [19]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print(f"Target classes (genres): {label_encoder.classes_}\n")

Target classes (genres): ['Action' 'Action-Adventure' 'Adventure' 'Board Game' 'Education'
 'Fighting' 'MMO' 'Misc' 'Music' 'Party' 'Platform' 'Puzzle' 'Racing'
 'Role-Playing' 'Sandbox' 'Shooter' 'Simulation' 'Sports' 'Strategy'
 'Visual Novel']



Step 4- Preprocessing Pipeline (Imputation and Encoding)

In [34]:
preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='mean'), NUMERICAL_FEATURES),
    ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), CATEGORICAL_FEATURES)
], remainder='drop')

X_preprocessed = pd.DataFrame(preprocessor.fit_transform(X),
                              columns=NUMERICAL_FEATURES + list(preprocessor.named_transformers_['cat'].get_feature_names_out(CATEGORICAL_FEATURES)))

print(X_preprocessed)

       critic_score  console_2600  console_3DO  console_3DS  console_5200  \
0           9.40000           0.0          0.0          0.0           0.0   
1           9.70000           0.0          0.0          0.0           0.0   
2           9.60000           0.0          0.0          0.0           0.0   
3           7.22044           0.0          0.0          0.0           0.0   
4           8.10000           0.0          0.0          0.0           0.0   
...             ...           ...          ...          ...           ...   
64011       7.22044           0.0          0.0          0.0           0.0   
64012       7.22044           0.0          0.0          0.0           0.0   
64013       7.22044           0.0          0.0          0.0           0.0   
64014       7.22044           0.0          0.0          0.0           0.0   
64015       7.22044           0.0          0.0          0.0           0.0   

       console_7800  console_ACPC  console_AJ  console_AST  console_Aco  ..

Step 5- Data Splitting

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)

print(f"Original features shape: {X.shape}")
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples\n")

Original features shape: (64016, 2)
Training set size: 44811 samples
Testing set size: 19205 samples



step 6- Model definition and training

In [38]:
model_pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', DecisionTreeClassifier(max_depth=10, random_state=42))])
print("Starting model training..."); model_pipeline.fit(X_train, y_train); print("Model training complete.\n")

Starting model training...
Model training complete.



Step 7- Predicition and Evaluation

In [41]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, (y_pred := model_pipeline.predict(X_test))))

new_data = pd.DataFrame([['PS4', 9.5], ['Wii', 5.0]], columns=FEATURE_COLUMNS)
for i, row in enumerate(new_data.values):
    print(f"New Data {i+1} {tuple(row)}: {label_encoder.inverse_transform(model_pipeline.predict(pd.DataFrame([row], columns=FEATURE_COLUMNS)))[0]}")


Accuracy: 0.17537099713616247
New Data 1 ('PS4', 9.5): Action
New Data 2 ('Wii', 5.0): Action
